
# Tekst jako dane: TF-IDF, embeddings i topic modelling

Notebook obejmuje dwa scenariusze:

## Część A — Online Retail

Krótkie nazwy produktów:

```text
Description
→ reprezentacja tekstu
→ podobieństwo
→ podobne produkty
→ clustering
→ kontrola jakości katalogu
```

## Część B — Consumer Complaints

Dłuższe narracje klientów:

```text
complaint narrative
→ czyszczenie
→ TF-IDF
→ NMF
→ tematy
→ embeddings / wektory semantyczne
→ semantic search
→ clustering
→ materiał do podsumowania przez LLM
```

Notebook ma dwa tryby reprezentacji semantycznej:

1. **Domyślny tryb offline:** TF-IDF + TruncatedSVD, czyli latent semantic vectors.
2. **Opcjonalny tryb neural embeddings:** `sentence-transformers`.

Domyślny tryb nie wymaga pobierania modelu z internetu. Neural embeddings są zalecanym wariantem do właściwego demo semantycznego.


In [ ]:

# W środowisku lokalnym odkomentuj w razie potrzeby:
# %pip install -U polars pandas scikit-learn matplotlib
#
# Opcjonalne neural embeddings i BERTopic:
# %pip install -U sentence-transformers bertopic umap-learn hdbscan

from __future__ import annotations

from pathlib import Path
from typing import Iterable
import importlib.util
import re

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import Normalizer

print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)



## Konfiguracja

Ustaw ścieżki do właściwych danych.

Popularne kolumny:

### Online Retail

- `StockCode`
- `Description`

### Consumer Complaints

- `Consumer complaint narrative`
- `Product`
- `Issue`
- `Sub-issue`
- `Date received`

Jeżeli plików nie ma, notebook tworzy dane demonstracyjne.


In [ ]:

ONLINE_RETAIL_PATH = Path("data/online_retail.parquet")
COMPLAINTS_PATH = Path("data/consumer_complaints.csv")

CREATE_DEMO_IF_MISSING = True

# Neural embeddings wymagają pakietu i lokalnego modelu albo dostępu do internetu
# przy pierwszym uruchomieniu.
USE_NEURAL_EMBEDDINGS = False
SENTENCE_TRANSFORMER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# BERTopic jest opcjonalny i domyślnie wyłączony.
RUN_BERTOPIC = False

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



# Część A. Krótkie teksty produktowe — Online Retail



## 1. Dane produktowe

Właściwy zbiór Online Retail zawiera nazwy produktów w kolumnie `Description`.  
Do analizy tekstowej potrzebujemy tabeli unikalnych produktów, a nie wszystkich pozycji transakcyjnych.


In [ ]:

DEMO_PRODUCT_GROUPS = {
    "bags": [
        "JUMBO BAG RED RETROSPOT",
        "JUMBO BAG PINK POLKADOT",
        "LUNCH BAG SPACEBOY DESIGN",
        "LUNCH BAG RED RETROSPOT",
        "SHOPPING BAG VINTAGE ROSE",
        "COTTON SHOPPING BAG BLUE STRIPE",
        "LARGE TOTE BAG UNION JACK",
        "SMALL POLKADOT SHOPPING BAG",
    ],
    "candles": [
        "WHITE HANGING HEART T-LIGHT HOLDER",
        "GLASS STAR FROSTED T-LIGHT HOLDER",
        "SILVER HANGING HEART CANDLE HOLDER",
        "SET OF THREE GLASS CANDLE HOLDERS",
        "VINTAGE GLASS CANDLE LANTERN",
        "RED METAL HEART T-LIGHT HOLDER",
        "CERAMIC STAR CANDLE HOLDER",
        "SMALL GLASS LANTERN",
    ],
    "kitchen": [
        "SET OF THREE CAKE TINS PANTRY DESIGN",
        "CERAMIC MUG BLUE POLKADOT",
        "VINTAGE KITCHEN STORAGE JAR",
        "WOODEN SPOON SET",
        "RED RETROSPOT TEA TOWEL",
        "ENAMEL BREAD BIN CREAM",
        "GLASS STORAGE JAR WITH LID",
        "KITCHEN APRON VINTAGE ROSE",
    ],
    "frames": [
        "WOODEN PICTURE FRAME WHITE FINISH",
        "VINTAGE PHOTO FRAME GOLD",
        "SMALL SILVER PHOTO FRAME",
        "DISTRESSED WOODEN PICTURE FRAME",
        "HEART SHAPED PHOTO FRAME",
        "BLACK ORNATE PICTURE FRAME",
        "WHITE WOOD PHOTO FRAME",
        "MINI FRAME VINTAGE ROSE",
    ],
    "christmas": [
        "CHRISTMAS TREE DECORATION SILVER",
        "WOODEN STAR CHRISTMAS DECORATION",
        "RED FELT CHRISTMAS STOCKING",
        "SET OF CHRISTMAS TREE BAUBLES",
        "WHITE WOODEN CHRISTMAS ANGEL",
        "CHRISTMAS GARLAND RED STARS",
        "VINTAGE CHRISTMAS GIFT TAGS",
        "SMALL CHRISTMAS REINDEER DECORATION",
    ],
    "children": [
        "WOODEN TOY TRAIN SET",
        "CHILDRENS APRON SPACEBOY",
        "PINK RABBIT SOFT TOY",
        "WOODEN PUZZLE ALPHABET",
        "CHILDRENS LUNCH BOX DINOSAUR",
        "SET OF COLOURING PENCILS",
        "MINI WOODEN DOLL HOUSE",
        "CHILDRENS GARDENING SET",
    ],
}


def create_demo_products() -> pd.DataFrame:
    rows = []
    product_id = 1000

    for group, descriptions in DEMO_PRODUCT_GROUPS.items():
        for description in descriptions:
            rows.append(
                {
                    "stock_code": f"P{product_id}",
                    "description": description,
                    "demo_category": group,
                }
            )
            product_id += 1

    # Celowo podobne i problematyczne nazwy do kontroli jakości.
    rows.extend(
        [
            {
                "stock_code": "P9001",
                "description": "JUMBO BAG RED RETRO SPOT",
                "demo_category": "bags",
            },
            {
                "stock_code": "P9002",
                "description": "WHITE HANGING HEART TLIGHT HOLDER",
                "demo_category": "candles",
            },
            {
                "stock_code": "P9003",
                "description": "UNKNOWN PRODUCT",
                "demo_category": "unknown",
            },
        ]
    )

    return pd.DataFrame(rows)


def normalized_column_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", name.lower())


def load_online_retail_products(path: Path) -> pd.DataFrame:
    if not path.exists():
        if not CREATE_DEMO_IF_MISSING:
            raise FileNotFoundError(path)

        print(
            "UWAGA: używany jest demonstracyjny katalog produktów. "
            "Do wykładu ustaw ONLINE_RETAIL_PATH."
        )
        return create_demo_products()

    if path.suffix.lower() == ".parquet":
        frame = pl.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        frame = pl.read_csv(path, infer_schema_length=10_000)
    else:
        raise ValueError("Online Retail: obsługiwane formaty CSV i Parquet.")

    lookup = {
        normalized_column_name(column): column
        for column in frame.columns
    }

    stock_column = lookup.get("stockcode")
    description_column = lookup.get("description")

    if description_column is None:
        raise ValueError(
            f"Nie znaleziono kolumny Description. Kolumny: {frame.columns}"
        )

    selected = [
        pl.col(description_column).cast(pl.String).alias("description"),
    ]

    if stock_column:
        selected.insert(
            0,
            pl.col(stock_column).cast(pl.String).alias("stock_code"),
        )
    else:
        selected.insert(
            0,
            pl.col(description_column)
            .cast(pl.String)
            .alias("stock_code"),
        )

    products = (
        frame.lazy()
        .select(selected)
        .drop_nulls(["description"])
        .with_columns(
            pl.col("description")
            .str.strip_chars()
            .str.to_uppercase()
        )
        .filter(
            pl.col("description").str.len_chars() >= 3
        )
        .unique(subset=["stock_code", "description"])
        .collect()
        .to_pandas()
    )

    return products


products = load_online_retail_products(ONLINE_RETAIL_PATH)
print("Unikalne produkty:", len(products))
products.head()



## 2. TF-IDF jako reprezentacja leksykalna

TF-IDF nadaje wysokie wagi słowom charakterystycznym dla dokumentu i niższe słowom pojawiającym się bardzo często.

Dla krótkich nazw produktów dobrze sprawdzają się także bigramy, np.:

- `photo frame`,
- `candle holder`,
- `lunch bag`.

TF-IDF porównuje przede wszystkim słowa i frazy, a nie pełne znaczenie semantyczne.


In [ ]:

product_tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    stop_words="english",
)

X_products_tfidf = product_tfidf.fit_transform(
    products["description"]
)

print("Macierz TF-IDF:", X_products_tfidf.shape)


In [ ]:

def find_similar_from_matrix(
    query: str,
    vectorizer: TfidfVectorizer,
    document_matrix,
    documents: pd.DataFrame,
    text_column: str,
    top_k: int = 5,
) -> pd.DataFrame:
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(
        query_vector,
        document_matrix,
    ).ravel()

    top_indices = np.argsort(scores)[::-1][:top_k]

    result = documents.iloc[top_indices].copy()
    result["similarity"] = scores[top_indices]

    return result


find_similar_from_matrix(
    query="red reusable shopping bag",
    vectorizer=product_tfidf,
    document_matrix=X_products_tfidf,
    documents=products,
    text_column="description",
    top_k=8,
)



## 3. Wektory semantyczne

### Tryb offline

TF-IDF + `TruncatedSVD` tworzy gęste wektory latent semantic analysis.  
Nie są to neural embeddings, ale pozwalają wykonać podobieństwo i clustering bez pobierania modelu.

### Neural embeddings

`SentenceTransformer.encode()` zamienia każdy tekst na gęsty wektor o stałej długości.  
Modele sentence-transformers są projektowane m.in. do semantic similarity, semantic search i clusteringu.

Kod automatycznie wybierze neural embeddings, gdy:

- `USE_NEURAL_EMBEDDINGS=True`,
- pakiet `sentence-transformers` jest zainstalowany,
- model jest dostępny lokalnie lub może zostać pobrany.


In [ ]:

def build_semantic_vectors(
    texts: list[str],
    tfidf_matrix,
    use_neural_embeddings: bool,
) -> tuple[np.ndarray, str, object | None]:
    sentence_transformers_available = (
        importlib.util.find_spec("sentence_transformers") is not None
    )

    if use_neural_embeddings and sentence_transformers_available:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(
            SENTENCE_TRANSFORMER_MODEL
        )
        vectors = model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        return (
            np.asarray(vectors),
            f"SentenceTransformer: {SENTENCE_TRANSFORMER_MODEL}",
            model,
        )

    max_components = min(
        64,
        tfidf_matrix.shape[0] - 1,
        tfidf_matrix.shape[1] - 1,
    )

    if max_components < 2:
        raise ValueError(
            "Za mało danych do utworzenia wektorów SVD."
        )

    svd = TruncatedSVD(
        n_components=max_components,
        random_state=42,
    )
    normalizer = Normalizer(copy=False)

    vectors = svd.fit_transform(tfidf_matrix)
    vectors = normalizer.fit_transform(vectors)

    return (
        vectors,
        "TF-IDF + TruncatedSVD — offline fallback",
        (svd, normalizer),
    )


product_vectors, product_vector_method, product_vector_model = (
    build_semantic_vectors(
        products["description"].tolist(),
        X_products_tfidf,
        USE_NEURAL_EMBEDDINGS,
    )
)

print(product_vector_method)
print("Kształt wektorów:", product_vectors.shape)


In [ ]:

def encode_query(
    query: str,
    method_name: str,
    model_or_transformers,
    tfidf_vectorizer: TfidfVectorizer,
) -> np.ndarray:
    if method_name.startswith("SentenceTransformer"):
        return np.asarray(
            model_or_transformers.encode(
                [query],
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        )

    svd, normalizer = model_or_transformers
    query_tfidf = tfidf_vectorizer.transform([query])
    query_vector = svd.transform(query_tfidf)
    return normalizer.transform(query_vector)


def semantic_search(
    query: str,
    documents: pd.DataFrame,
    document_vectors: np.ndarray,
    text_column: str,
    method_name: str,
    model_or_transformers,
    tfidf_vectorizer: TfidfVectorizer,
    top_k: int = 5,
) -> pd.DataFrame:
    query_vector = encode_query(
        query,
        method_name,
        model_or_transformers,
        tfidf_vectorizer,
    )

    scores = cosine_similarity(
        query_vector,
        document_vectors,
    ).ravel()

    top_indices = np.argsort(scores)[::-1][:top_k]

    result = documents.iloc[top_indices].copy()
    result["similarity"] = scores[top_indices]
    return result


semantic_search(
    query="decorative holder for a small candle",
    documents=products,
    document_vectors=product_vectors,
    text_column="description",
    method_name=product_vector_method,
    model_or_transformers=product_vector_model,
    tfidf_vectorizer=product_tfidf,
    top_k=8,
)



## 4. Clustering produktów

Grupujemy wektory produktów przez K-Means.  
Właściwy model może znaleźć grupy odpowiadające kategoriom takim jak:

- torby,
- świeczniki,
- artykuły kuchenne,
- ramki,
- dekoracje świąteczne,
- produkty dziecięce.

Algorytm zwróci jednak tylko techniczne numery klastrów.


In [ ]:

PRODUCT_CLUSTERS = min(6, max(2, len(products) // 8))

product_kmeans = KMeans(
    n_clusters=PRODUCT_CLUSTERS,
    n_init=20,
    random_state=42,
)

products["product_cluster_id"] = (
    product_kmeans.fit_predict(product_vectors)
)

product_cluster_sizes = (
    products.groupby("product_cluster_id")
    .size()
    .rename("products")
    .reset_index()
)

product_cluster_sizes


In [ ]:

def show_cluster_examples(
    frame: pd.DataFrame,
    cluster_column: str,
    text_column: str,
    examples_per_cluster: int = 6,
) -> pd.DataFrame:
    rows = []

    for cluster_id, group in frame.groupby(cluster_column):
        examples = group[text_column].head(
            examples_per_cluster
        ).tolist()

        rows.append(
            {
                "cluster_id": cluster_id,
                "records": len(group),
                "examples": " | ".join(examples),
            }
        )

    return pd.DataFrame(rows).sort_values("cluster_id")


product_cluster_examples = show_cluster_examples(
    products.sort_values(
        ["product_cluster_id", "description"]
    ),
    cluster_column="product_cluster_id",
    text_column="description",
)

product_cluster_examples



## 5. Kontrola jakości katalogu

Możliwe zastosowania podobieństwa:

- wykrycie niemal identycznych nazw,
- znalezienie różnych kodów opisujących ten sam produkt,
- wskazanie produktu niepasującego do kategorii,
- znalezienie opisów z literówkami,
- content-based recommendations.

Poniżej znajdujemy pary o najwyższym podobieństwie, pomijając porównanie produktu z samym sobą.


In [ ]:

product_similarity = cosine_similarity(product_vectors)
np.fill_diagonal(product_similarity, -1)

candidate_pairs = []

for row_index in range(len(products)):
    best_match_index = int(
        np.argmax(product_similarity[row_index])
    )

    candidate_pairs.append(
        {
            "product_a": products.iloc[row_index]["description"],
            "product_b": products.iloc[best_match_index]["description"],
            "similarity": product_similarity[
                row_index,
                best_match_index,
            ],
        }
    )

similar_product_pairs = (
    pd.DataFrame(candidate_pairs)
    .sort_values("similarity", ascending=False)
    .drop_duplicates(
        subset=["product_a", "product_b"]
    )
)

similar_product_pairs.head(12)



# Część B. Dłuższe teksty — Consumer Complaints



## 6. Dane demonstracyjne i wczytanie zbioru

Consumer Complaint Database może zawierać narracyjne opisy problemów napisane przez konsumentów oraz pola strukturalne, takie jak produkt i rodzaj problemu.

Do właściwego topic modelling potrzebujemy dłuższych wypowiedzi, a nie wyłącznie krótkich kategorii.


In [ ]:

DEMO_COMPLAINT_TOPICS = {
    "credit_reporting": [
        "My credit report contains an account that does not belong to me and the bureau keeps verifying it without providing evidence.",
        "I disputed an incorrect late payment on my credit file but the information was not removed.",
        "The credit reporting company lists the same debt twice and my score has fallen.",
        "An address and employer that I never used appear on my credit report.",
    ],
    "debt_collection": [
        "A debt collector is calling me about a debt I do not recognize and refuses to send validation.",
        "The collection company continues contacting my family after I asked it to stop.",
        "I paid the account but the collector still reports an outstanding balance.",
        "The collector threatened legal action for a debt that is past the statute of limitations.",
    ],
    "card_transactions": [
        "My card was charged for purchases I did not authorize and the bank denied my dispute.",
        "A merchant refund never appeared on my credit card statement.",
        "The bank blocked my card while I was traveling and I could not access my funds.",
        "I was charged a recurring subscription after cancelling the service.",
    ],
    "mortgage": [
        "My mortgage servicer applied my payment to fees instead of principal and reported me late.",
        "The company lost my loan modification documents several times.",
        "My escrow payment increased without a clear explanation.",
        "The servicer started foreclosure even though my assistance application was under review.",
    ],
    "bank_account": [
        "My checking account was closed without warning and the remaining balance has not been returned.",
        "The bank charged repeated overdraft fees even though the transactions were pending.",
        "A cash deposit is missing from my account and the bank cannot locate it.",
        "I cannot access online banking after the institution changed its verification process.",
    ],
    "student_loan": [
        "My student loan payment was placed on the wrong loan and interest continued to accrue.",
        "The servicer gave me incorrect information about an income driven repayment plan.",
        "My qualifying public service payments are missing from the account history.",
        "The company placed the loan in forbearance without my permission.",
    ],
}


def create_demo_complaints(
    repetitions_per_template: int = 6,
) -> pd.DataFrame:
    rows = []
    complaint_id = 1
    dates = pd.date_range(
        "2025-01-01",
        periods=500,
        freq="D",
    )

    product_map = {
        "credit_reporting": "Credit reporting",
        "debt_collection": "Debt collection",
        "card_transactions": "Credit card",
        "mortgage": "Mortgage",
        "bank_account": "Checking or savings account",
        "student_loan": "Student loan",
    }

    issue_map = {
        "credit_reporting": "Incorrect information on report",
        "debt_collection": "Attempts to collect debt not owed",
        "card_transactions": "Problem with a purchase or transfer",
        "mortgage": "Trouble during payment process",
        "bank_account": "Managing an account",
        "student_loan": "Dealing with the lender or servicer",
    }

    modifiers = [
        "I contacted customer service twice but received no useful response.",
        "This problem has continued for several weeks.",
        "I have documents that support my complaint.",
        "The company closed my case without resolving the issue.",
        "I am requesting a written explanation and correction.",
        "The situation is causing financial stress and additional fees.",
    ]

    for topic, templates in DEMO_COMPLAINT_TOPICS.items():
        for template_index, template in enumerate(templates):
            for repetition in range(repetitions_per_template):
                modifier = modifiers[
                    (template_index + repetition) % len(modifiers)
                ]
                rows.append(
                    {
                        "complaint_id": complaint_id,
                        "date_received": dates[
                            complaint_id % len(dates)
                        ],
                        "product": product_map[topic],
                        "issue": issue_map[topic],
                        "consumer_complaint_narrative": (
                            f"{template} {modifier}"
                        ),
                        "demo_topic": topic,
                    }
                )
                complaint_id += 1

    return pd.DataFrame(rows)


def snake_case(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


def load_complaints(path: Path) -> pd.DataFrame:
    if not path.exists():
        if not CREATE_DEMO_IF_MISSING:
            raise FileNotFoundError(path)

        print(
            "UWAGA: używany jest demonstracyjny zbiór skarg. "
            "Do wykładu ustaw COMPLAINTS_PATH."
        )
        return create_demo_complaints()

    if path.suffix.lower() == ".parquet":
        frame = pl.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        frame = pl.read_csv(
            path,
            infer_schema_length=20_000,
            ignore_errors=True,
        )
    else:
        raise ValueError(
            "Consumer Complaints: obsługiwane formaty CSV i Parquet."
        )

    rename_map = {
        column: snake_case(column)
        for column in frame.columns
    }
    frame = frame.rename(rename_map)

    narrative_candidates = [
        "consumer_complaint_narrative",
        "complaint_narrative",
        "narrative",
    ]

    narrative_column = next(
        (
            column
            for column in narrative_candidates
            if column in frame.columns
        ),
        None,
    )

    if narrative_column is None:
        raise ValueError(
            "Nie znaleziono kolumny Consumer complaint narrative."
        )

    select_expressions = [
        pl.col(narrative_column)
        .cast(pl.String)
        .alias("consumer_complaint_narrative"),
    ]

    for optional_column in [
        "complaint_id",
        "date_received",
        "product",
        "issue",
        "sub_issue",
        "company",
    ]:
        if optional_column in frame.columns:
            select_expressions.append(
                pl.col(optional_column)
            )

    cleaned = (
        frame.lazy()
        .select(select_expressions)
        .drop_nulls(["consumer_complaint_narrative"])
        .with_columns(
            pl.col("consumer_complaint_narrative")
            .str.strip_chars()
        )
        .filter(
            pl.col("consumer_complaint_narrative")
            .str.len_chars()
            >= 40
        )
        .collect()
        .to_pandas()
    )

    return cleaned


complaints = load_complaints(COMPLAINTS_PATH)
print("Skargi z narracją:", len(complaints))
complaints.head()



## 7. Minimalne czyszczenie tekstu

Dla nowoczesnych modeli embeddings zwykle zachowujemy naturalną treść zdania.

Dla klasycznego TF-IDF możemy:

- zmienić tekst na małe litery,
- zastąpić sekwencje białych znaków,
- opcjonalnie usunąć bardzo częste szablonowe fragmenty,
- pozostawić podstawową tokenizację biblioteki.

Nie wykonujemy agresywnego usuwania słów bez sprawdzenia, ponieważ negacje i kontekst mogą mieć znaczenie.


In [ ]:

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9$%\-\s']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


complaints["clean_text"] = (
    complaints["consumer_complaint_narrative"]
    .map(clean_text)
)

complaints[
    [
        "consumer_complaint_narrative",
        "clean_text",
    ]
].head(3)



## 8. TF-IDF dla narracji

Ograniczamy liczbę cech i ignorujemy bardzo rzadkie oraz niemal powszechne terminy.

Bigramy pomagają zachować wyrażenia takie jak:

- `credit report`,
- `debt collector`,
- `student loan`,
- `customer service`.


In [ ]:

complaint_tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=8_000,
    sublinear_tf=True,
)

X_complaints_tfidf = complaint_tfidf.fit_transform(
    complaints["clean_text"]
)

print("Macierz TF-IDF:", X_complaints_tfidf.shape)



## 9. Topic modelling przez NMF

NMF rozkłada nieujemną macierz dokument–termin na:

- reprezentację dokumentów przez tematy,
- reprezentację tematów przez słowa.

Temat nie otrzymuje automatycznie biznesowej nazwy. Interpretujemy go na podstawie terminów o najwyższych wagach oraz przykładowych dokumentów.


In [ ]:

N_TOPICS = min(
    6,
    max(2, len(complaints) // 20),
)

nmf_model = NMF(
    n_components=N_TOPICS,
    init="nndsvda",
    random_state=42,
    max_iter=500,
)

document_topic_weights = nmf_model.fit_transform(
    X_complaints_tfidf
)

feature_names = np.array(
    complaint_tfidf.get_feature_names_out()
)


def top_words_for_topics(
    model: NMF,
    vocabulary: np.ndarray,
    top_n: int = 10,
) -> pd.DataFrame:
    rows = []

    for topic_id, topic_weights in enumerate(
        model.components_
    ):
        top_indices = topic_weights.argsort()[::-1][:top_n]
        rows.append(
            {
                "topic_id": topic_id,
                "top_terms": ", ".join(
                    vocabulary[top_indices]
                ),
            }
        )

    return pd.DataFrame(rows)


nmf_topics = top_words_for_topics(
    nmf_model,
    feature_names,
    top_n=10,
)

nmf_topics


In [ ]:

complaints["nmf_topic_id"] = (
    document_topic_weights.argmax(axis=1)
)
complaints["nmf_topic_strength"] = (
    document_topic_weights.max(axis=1)
)

nmf_topic_summary = (
    complaints.groupby("nmf_topic_id")
    .agg(
        complaints=(
            "consumer_complaint_narrative",
            "size",
        ),
        example=(
            "consumer_complaint_narrative",
            "first",
        ),
        average_strength=(
            "nmf_topic_strength",
            "mean",
        ),
    )
    .reset_index()
    .merge(
        nmf_topics,
        left_on="nmf_topic_id",
        right_on="topic_id",
        how="left",
    )
    .drop(columns=["topic_id"])
)

nmf_topic_summary



## 10. Embeddings / wektory semantyczne dla skarg

Tak jak dla produktów:

- neural embeddings są preferowanym wariantem semantycznym,
- tryb offline wykorzystuje TF-IDF + TruncatedSVD.

Następnie wektory można wykorzystać do:

- semantic search,
- clusteringu,
- klasyfikacji,
- wykrywania nowych i nietypowych tematów.


In [ ]:

complaint_vectors, complaint_vector_method, complaint_vector_model = (
    build_semantic_vectors(
        complaints["clean_text"].tolist(),
        X_complaints_tfidf,
        USE_NEURAL_EMBEDDINGS,
    )
)

print(complaint_vector_method)
print("Kształt wektorów:", complaint_vectors.shape)



## 11. Semantic search

Wyszukiwanie słów kluczowych wymaga podobnych słów w zapytaniu i dokumencie.

Wyszukiwanie semantyczne porównuje wektory. Może znaleźć podobne przypadki opisane innym językiem.


In [ ]:

semantic_search(
    query=(
        "The bank did not return money after "
        "an unauthorized card payment"
    ),
    documents=complaints,
    document_vectors=complaint_vectors,
    text_column="consumer_complaint_narrative",
    method_name=complaint_vector_method,
    model_or_transformers=complaint_vector_model,
    tfidf_vectorizer=complaint_tfidf,
    top_k=8,
)[
    [
        column
        for column in [
            "product",
            "issue",
            "consumer_complaint_narrative",
            "similarity",
        ]
        if column in complaints.columns
        or column in {"similarity"}
    ]
]



## 12. Clustering wektorów tekstu

K-Means grupuje narracje o podobnej reprezentacji.

W praktyce można użyć również HDBSCAN, który:

- nie wymaga liczby klastrów,
- może oznaczyć część dokumentów jako szum,
- jest często wykorzystywany w pipeline BERTopic.


In [ ]:

candidate_topic_counts = range(
    2,
    min(9, len(complaints) - 1),
)

embedding_cluster_evaluation = []

for topic_count in candidate_topic_counts:
    model = KMeans(
        n_clusters=topic_count,
        n_init=20,
        random_state=42,
    )
    labels = model.fit_predict(complaint_vectors)

    score = silhouette_score(
        complaint_vectors,
        labels,
    )

    embedding_cluster_evaluation.append(
        {
            "clusters": topic_count,
            "silhouette_score": score,
        }
    )

embedding_cluster_evaluation = pd.DataFrame(
    embedding_cluster_evaluation
)

embedding_cluster_evaluation


In [ ]:

recommended_embedding_clusters = int(
    embedding_cluster_evaluation.sort_values(
        "silhouette_score",
        ascending=False,
    ).iloc[0]["clusters"]
)

EMBEDDING_CLUSTERS = recommended_embedding_clusters

complaint_kmeans = KMeans(
    n_clusters=EMBEDDING_CLUSTERS,
    n_init=20,
    random_state=42,
)

complaints["embedding_cluster_id"] = (
    complaint_kmeans.fit_predict(complaint_vectors)
)

print(
    "Liczba klastrów:",
    EMBEDDING_CLUSTERS,
)


In [ ]:

embedding_cluster_examples = show_cluster_examples(
    complaints.sort_values(
        [
            "embedding_cluster_id",
            "consumer_complaint_narrative",
        ]
    ),
    cluster_column="embedding_cluster_id",
    text_column="consumer_complaint_narrative",
    examples_per_cluster=4,
)

embedding_cluster_examples



## 13. Wizualizacja dokumentów

Redukujemy wektory do dwóch wymiarów wyłącznie w celu wizualizacji.

Clustering został wykonany na pełnych wektorach, a nie na samym wykresie 2D.


In [ ]:

visual_components = min(
    2,
    complaint_vectors.shape[1],
)

if visual_components == 2:
    complaint_visualizer = TruncatedSVD(
        n_components=2,
        random_state=42,
    )
    complaint_vectors_2d = (
        complaint_visualizer.fit_transform(
            complaint_vectors
        )
    )

    plt.figure(figsize=(8.5, 6))
    scatter = plt.scatter(
        complaint_vectors_2d[:, 0],
        complaint_vectors_2d[:, 1],
        c=complaints[
            "embedding_cluster_id"
        ].to_numpy(),
        s=22,
        alpha=0.7,
    )
    plt.title(
        "Klastry narracji w projekcji do dwóch wymiarów"
    )
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.legend(
        *scatter.legend_elements(),
        title="cluster_id",
    )
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()



## 14. Porównanie podejść topic modelling

### LDA

- model probabilistyczny,
- zwykle pracuje na liczbach wystąpień słów,
- dokument może być mieszaniną tematów.

### NMF

- dobrze współpracuje z TF-IDF,
- daje addytywne tematy reprezentowane przez ważne słowa,
- jest prosty i szybki dla klasycznego demo.

### Embeddings + clustering

- grupuje dokumenty na podstawie reprezentacji semantycznej,
- lepiej radzi sobie z różnymi sformułowaniami,
- nazwy tematów trzeba wydobyć z przykładów lub terminów.

### BERTopic

Typowy przepływ:

```text
embeddings
→ redukcja wymiarów
→ clustering
→ c-TF-IDF
→ reprezentacja tematów
```

Poniższa komórka jest opcjonalna.


In [ ]:

if RUN_BERTOPIC:
    if importlib.util.find_spec("bertopic") is None:
        raise ImportError(
            "Zainstaluj: sentence-transformers bertopic "
            "umap-learn hdbscan"
        )

    from bertopic import BERTopic

    topic_model = BERTopic(
        language="english",
        calculate_probabilities=False,
        verbose=False,
    )

    bertopic_topics, _ = topic_model.fit_transform(
        complaints["consumer_complaint_narrative"].tolist()
    )

    display(topic_model.get_topic_info().head(15))
else:
    print(
        "BERTopic jest wyłączony. "
        "Ustaw RUN_BERTOPIC=True po instalacji zależności."
    )



## 15. Routing i klasyfikacja tekstu

Jeżeli posiadamy historyczną kolumnę `Product` lub `Issue`, temat może stać się targetem klasyfikacji:

```text
narracja
→ TF-IDF lub embeddings
→ klasyfikator
→ przewidywany zespół / kategoria
```

Topic modelling jest zadaniem bez nadzoru — odkrywa strukturę bez gotowych etykiet.

Routing jest zadaniem nadzorowanym — model uczy się istniejących kategorii.



## 16. Materiał do podsumowania przez LLM

LLM nie powinien otrzymywać całej surowej bazy i samodzielnie liczyć statystyk.

Najpierw kod oblicza:

- liczbę dokumentów w klastrze,
- zmianę liczebności w czasie,
- najważniejsze terminy,
- reprezentatywne przykłady.

Dopiero te wyniki przekazujemy do LLM, prosząc o:

- nazwę tematu,
- krótkie podsumowanie,
- interpretację biznesową,
- rekomendowane następne kroki.


In [ ]:

def build_llm_topic_prompt(
    topic_row: pd.Series,
    examples: Iterable[str],
) -> str:
    examples_text = "\n".join(
        f"- {example}"
        for example in list(examples)[:5]
    )

    return f"""
Jesteś analitykiem jakości obsługi klienta.

Na podstawie zagregowanych informacji nadaj tematowi:
1. krótką nazwę,
2. opis w 2-3 zdaniach,
3. możliwy wpływ biznesowy,
4. jedną rekomendację dalszej analizy.

Liczba zgłoszeń: {topic_row['complaints']}
Najważniejsze terminy: {topic_row['top_terms']}

Przykładowe zgłoszenia:
{examples_text}

Nie zmieniaj podanych liczb i nie dopisuj statystyk,
których nie ma w danych.
""".strip()


sample_topic_id = int(
    nmf_topic_summary.sort_values(
        "complaints",
        ascending=False,
    ).iloc[0]["nmf_topic_id"]
)

sample_topic_row = nmf_topic_summary[
    nmf_topic_summary["nmf_topic_id"]
    == sample_topic_id
].iloc[0]

sample_examples = complaints[
    complaints["nmf_topic_id"] == sample_topic_id
]["consumer_complaint_narrative"].head(5)

llm_prompt = build_llm_topic_prompt(
    sample_topic_row,
    sample_examples,
)

print(llm_prompt)



## 17. Przykładowa struktura odpowiedzi LLM

W notebooku nie wywołujemy konkretnego API. Dzięki temu przykład pozostaje niezależny od dostawcy.

Przykładowa oczekiwana odpowiedź:

```text
Nazwa tematu:
Błędne informacje w raportach kredytowych

Podsumowanie:
Zgłoszenia dotyczą nieprawidłowych kont, płatności i danych osobowych,
które pozostają w raportach mimo wcześniejszych sporów.

Wpływ biznesowy:
Rosnąca liczba takich przypadków może zwiększać liczbę ponownych kontaktów
oraz ryzyko eskalacji.

Rekomendacja:
Porównać liczbę zgłoszeń i czas rozwiązania według firmy oraz miesiąca.
```

Obliczenia i liczebności pochodzą z kodu. Model językowy tworzy warstwę narracyjną.



## 18. Zapis wyników


In [ ]:

PRODUCTS_OUTPUT_PATH = (
    OUTPUT_DIR / "online_retail_product_text_clusters.parquet"
)
COMPLAINTS_OUTPUT_PATH = (
    OUTPUT_DIR / "consumer_complaints_topics.parquet"
)
TOPIC_SUMMARY_OUTPUT_PATH = (
    OUTPUT_DIR / "consumer_complaints_nmf_topic_summary.csv"
)

pl.from_pandas(products).write_parquet(
    PRODUCTS_OUTPUT_PATH
)
pl.from_pandas(complaints).write_parquet(
    COMPLAINTS_OUTPUT_PATH
)
nmf_topic_summary.to_csv(
    TOPIC_SUMMARY_OUTPUT_PATH,
    index=False,
)

print("Zapisano:")
print("-", PRODUCTS_OUTPUT_PATH)
print("-", COMPLAINTS_OUTPUT_PATH)
print("-", TOPIC_SUMMARY_OUTPUT_PATH)



# Podsumowanie

## Krótkie teksty produktowe

```text
Description
→ TF-IDF / embeddings
→ cosine similarity
→ podobne produkty
→ clustering
→ rekomendacje i kontrola jakości
```

## Dłuższe narracje

```text
tekst
→ TF-IDF
→ NMF
→ tematy i najważniejsze terminy
```

oraz:

```text
tekst
→ neural embeddings lub latent semantic vectors
→ semantic search
→ clustering
→ grupy znaczeniowe
```

## Rola LLM

```text
wyniki policzone przez kod
→ przykłady reprezentatywne
→ LLM
→ nazwa i biznesowe podsumowanie
```

Najważniejsza lekcja:

> Tekst staje się danymi dopiero po przekształceniu go do reprezentacji liczbowej. Wybór reprezentacji zależy od zadania: TF-IDF dobrze opisuje słowa, embeddings znaczenie, topic modelling strukturę korpusu, a LLM pomaga komunikować wcześniej obliczone wyniki.
